# Experimento 4: generación del pool de relaciones manualmente

En este notebook se construye un pool de relaciones a partir de la sintaxis, buscando root + verbo + auxiliar.

Para ello, se ejecutan estos pasos:

- Identificar el núcleo de la relación: El script localiza el ROOT. Si lleva auxiliar, une ambos para formar el bloque relacional.
- Buscar hacia la izquierda (Sujeto): El script busca la etiqueta nsubj y extrae el sintagma completo.
- Buscar hacia la derecha (Objeto/Complemento): El script busca el término introducido por la partícula y extrae el nombre propio resultante.


In [1]:
# Manipulación de datos
import pandas as pd

# Procesamiento lingüístico
import spacy

import re

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModel
from wordfreq import zipf_frequency
import torch

import hdbscan


In [2]:
# Cargamos el dataset limpio de DICAT.
# La columna que contiene el texto procesado es 'JUAN_RANA_cleaned'.

df = pd.read_csv("../Data/DicatCSVJuanRana_cleaned.csv")

print(f"Número de filas: {df.shape[0]}")
print(f"Número de columnas: {df.shape[1]}")

df.head()

Número de filas: 38
Número de columnas: 3


,YEAR,JUAN RANA,JUAN_RANA_cleaned
0,Introducción,"Aunque su verdadero nombre era Cosme Pérez, fu...","Aunque su verdadero nombre era Cosme Pérez, fu..."
1,1617,"Según Cotarelo (CM5, CLVIII), a quien sigue Sá...","Según Cotarelo , a quien sigue Sáez Raposo , C..."
2,1621,Consta la lista de la compañía de Juan Bautist...,Consta la lista de la compañía de Juan Bautist...
3,1624,Cosme Pérez aparece en la nómina de la compañí...,Cosme Pérez aparece en la nómina de la compañí...
4,1634,"Según Cotarelo, Cosme Pérez pertenecía a la co...","Según Cotarelo, Cosme Pérez pertenecía a la co..."


In [3]:
# Cargamos el modelo grande de spaCy para español.
# Este modelo permite tokenizar, etiquetar gramaticalmente y lematizar el texto.

nlp = spacy.load("es_core_news_lg")

In [ ]:
relations_raw = []
text = df["JUAN_RANA_cleaned"].iloc[0]

doc = nlp(str(text))

for sent in doc.sents:
    root = [token for token in sent if token.dep_ == "ROOT"]
    


Aunque su verdadero nombre era Cosme Pérez, fue conocido artísticamente como 'Juan Rana', según documenta ya la ***Genealogía***  —este apodo ya se le aplicaba en 1636—. Aunque Cotarelo aventuró que pudo haber nacido en Madrid , hoy sabemos que nació en Tudela de Duero (Valladolid), en cuya iglesia parroquial fue bautizado el 7 de abril de 1593. Fue hijo de Damián Pérez e Isabel de Basto, su segunda esposa, y fue el cuarto de una familia de cuatro hermanos . <br>Según algunas fuentes, como Sánchez Arjona y Subirá, estuvo casado con una actriz llamada Bernarda Ramírez . Por su lado, Rennert puso en duda su matrimonio con Bernarda Ramírez y creyó más probable que estuviese casado con Bernarda Manuela ['la Grifona'], que sería, según esto, su primera mujer . Sin embargo, ambos matrimonios no se encuentran documentados, como ya puso de relieve Cotarelo , y probablemente se fundan en deducciones realizadas a partir de algunos entremeses en los que Cosme Pérez figura junto con estas actrices

In [ ]:
def valid_lemma(word):

    freq = zipf_frequency(word, "es")

    return freq > 1

def safe_lemma(token):
    lemma = token.lemma_.lower().strip()

    # lema multipalabra raro
    if len(lemma.split()) > 1:
        return token.text.lower()

    # lema inexistente
    if not valid_lemma(lemma):
        return token.text.lower()

    return lemma

In [24]:
relations_raw = []

text = df["JUAN_RANA_cleaned"].iloc[0]
text = str(text).replace("<br>", ". ")

doc = nlp(text)

for sent in doc.sents:

    for token in sent:

        if token.pos_ == "VERB" and token.dep_ in [
            "ROOT", "ccomp", "xcomp", "advcl", "acl", "conj"
        ]:

            relation_parts = []

            prev_token = doc[token.i - 1] if token.i > 0 else None

            if (
                prev_token is not None
                and prev_token.pos_ == "AUX"
                and prev_token.lemma_ in ["ser", "estar"]
            ):
                relation_parts.append(prev_token.lemma_)
                relation_parts.append(token.text.lower())
            else:
                relation_parts.append(safe_lemma(token))

            found_connector = False

            # 1. Intentar primero con dependencia directa del verbo
            for child in token.children:
                if child.dep_ == "prep":
                    relation_parts.append(child.text.lower())
                    found_connector = True
                    break

            # 2. Si no hay dependencia, usar ventana corta a la derecha
            if not found_connector:
                for next_token in doc[token.i + 1 : min(token.i + 6, len(doc))]:

                    if next_token.pos_ in ["ADP", "SCONJ"]:
                        relation_parts.append(next_token.text.lower())
                        break

                    if next_token.is_punct:
                        break

            relation = " ".join(relation_parts)

            relations_raw.append({
                "relation": relation,
                "sentence": sent.text
            })

relations_raw[:50]

[{'relation': 'ser conocido como',
  'sentence': "Aunque su verdadero nombre era Cosme Pérez, fue conocido artísticamente como 'Juan Rana', según documenta ya la ***Genealogía***  —este apodo ya se le aplicaba en 1636—."},
 {'relation': 'documentar',
  'sentence': "Aunque su verdadero nombre era Cosme Pérez, fue conocido artísticamente como 'Juan Rana', según documenta ya la ***Genealogía***  —este apodo ya se le aplicaba en 1636—."},
 {'relation': 'aplicar en',
  'sentence': "Aunque su verdadero nombre era Cosme Pérez, fue conocido artísticamente como 'Juan Rana', según documenta ya la ***Genealogía***  —este apodo ya se le aplicaba en 1636—."},
 {'relation': 'aventurar que',
  'sentence': 'Aunque Cotarelo aventuró que pudo haber nacido en Madrid , hoy sabemos que nació en Tudela de Duero (Valladolid), en cuya iglesia parroquial fue bautizado el 7 de abril de 1593.'},
 {'relation': 'nacer en',
  'sentence': 'Aunque Cotarelo aventuró que pudo haber nacido en Madrid , hoy sabemos que na

In [17]:
relations_nominal = []

for sent in doc.sents:

    for token in sent:

        # Sustantivos/adjetivos con cópula
        has_cop = any(child.dep_ == "cop" for child in token.children)

        if has_cop and token.pos_ in ["NOUN", "ADJ"]:

            relation_parts = []

            # Añadir cópula lematizada
            for child in token.children:
                if child.dep_ == "cop":
                    relation_parts.append(safe_lemma(child))

            # Núcleo nominal/adjetival lematizado
            relation_parts.append(safe_lemma(token))

            # Buscar preposición cercana
            for next_token in doc[token.i + 1 : min(token.i + 8, len(doc))]:

                if next_token.pos_ in ["ADP", "SCONJ"]:
                    relation_parts.append(next_token.text)
                    break

                # Cortar en puntuación
                if next_token.is_punct:
                    break

            relation = " ".join(relation_parts)

            relations_nominal.append({
                "relation": relation,
                "sentence": sent.text
            })

print(relations_nominal[:50])

[{'relation': 'ser hijo de', 'sentence': 'Fue hijo de Damián Pérez e Isabel de Basto, su segunda esposa, y fue el cuarto de una familia de cuatro hermanos .'}, {'relation': 'ser cuarto de', 'sentence': 'Fue hijo de Damián Pérez e Isabel de Basto, su segunda esposa, y fue el cuarto de una familia de cuatro hermanos .'}, {'relation': 'estar casado con', 'sentence': 'Según algunas fuentes, como Sánchez Arjona y Subirá, estuvo casado con una actriz llamada Bernarda Ramírez .'}, {'relation': 'estar casado con', 'sentence': "Por su lado, Rennert puso en duda su matrimonio con Bernarda Ramírez y creyó más probable que estuviese casado con Bernarda Manuela ['la Grifona'], que sería, según esto, su primera mujer ."}, {'relation': 'ser mujer', 'sentence': "Por su lado, Rennert puso en duda su matrimonio con Bernarda Ramírez y creyó más probable que estuviese casado con Bernarda Manuela ['la Grifona'], que sería, según esto, su primera mujer ."}, {'relation': 'ser honbre de', 'sentence': 'Según l

In [20]:
relations_pool = relations_raw + relations_nominal
relations_df = pd.DataFrame(relations_pool)

relations_df.head()

,relation,sentence
0,ser conocido como,"Aunque su verdadero nombre era Cosme Pérez, fu..."
1,documentar,"Aunque su verdadero nombre era Cosme Pérez, fu..."
2,aplicar en,"Aunque su verdadero nombre era Cosme Pérez, fu..."
3,aventurar que,Aunque Cotarelo aventuró que pudo haber nacido...
4,nacer en,Aunque Cotarelo aventuró que pudo haber nacido...


In [10]:
relations_df["relation"].value_counts().head(50)

relation
figurar como        7
representar         6
tener               6
figurar             5
véase               5
hacer en            4
recoger de          4
suponer que         3
corresponder        3
aparecer en         3
escribir para       3
participar en       3
hacer de            3
llegar a            3
hacer a             2
hacer               2
observar            2
actuar en           2
véase para          2
creer               2
aludir a            2
saber que           2
nacer en            2
dar                 2
deducir que         2
ver                 2
hablar              2
poner en            2
salir a             2
encontrar           2
figurar con         2
anunciar en         2
señalar que         2
llevar de           2
estar casado con    2
representar a       2
intervenir en       2
ser anterior a      2
sugerir que         2
apuntar             2
obtener             2
representar en      2
contener de         2
identificar al      2
ocupar              2
d

In [11]:
relations_df = pd.DataFrame(relations_pool)

# Frecuencia de cada relación
freq = relations_df["relation"].value_counts().to_dict()

relations_df["freq"] = relations_df["relation"].map(freq)

# Número de palabras de la relación
relations_df["num_tokens"] = relations_df["relation"].apply(lambda x: len(str(x).split()))

# Si tiene preposición o marcador
relations_df["has_connector"] = relations_df["relation"].apply(
    lambda x: len(str(x).split()) > 1
)

# Verbo/núcleo inicial
relations_df["first_token"] = relations_df["relation"].apply(
    lambda x: str(x).split()[0] if len(str(x).split()) > 0 else ""
)

relations_df.head()

,relation,sentence,freq,num_tokens,has_connector,first_token
0,ser conocido como,"Aunque su verdadero nombre era Cosme Pérez, fu...",1,3,True,ser
1,documentar,"Aunque su verdadero nombre era Cosme Pérez, fu...",1,1,False,documentar
2,aplicar en,"Aunque su verdadero nombre era Cosme Pérez, fu...",1,2,True,aplicar
3,aventurar que,Aunque Cotarelo aventuró que pudo haber nacido...,1,2,True,aventurar
4,nacer en,Aunque Cotarelo aventuró que pudo haber nacido...,2,2,True,nacer


In [12]:
relations_df["weak_candidate"] = (
    (relations_df["num_tokens"] == 1) |
    (
        (relations_df["freq"] == 1) &
        (relations_df["num_tokens"] == 1)
    )
)

In [13]:
strong_relations = (
    relations_df[relations_df["weak_candidate"] == False]
    .groupby("relation")
    .agg(
        count=("relation", "count"),
        example=("sentence", "first")
    )
    .sort_values("count", ascending=False)
)

strong_relations.head(50)

,count,example
relation,,
figurar como,7,1) La primera y la segunda parte del entremés ...
recoger de,4,"Según la misma fuente, gozó de la confianza de..."
hacer en,4,"Como evoca, por su parte, M. L. Lobato, el luj..."
hacer de,3,"Sin embargo, ambos matrimonios no se encuentra..."
llegar a,3,"Como muestra de su popularidad, el autor de la..."
suponer que,3,Varey y Shergold recogen el juicio de Cotarelo...
participar en,3,"Según F. Sáez Raposo, la primera aparición esc..."
aparecer en,3,Dicho investigador señala que la participación...
escribir para,3,"Por su parte, Luis Quiñones de Benavente escri..."


In [14]:
weak_relations = (
    relations_df[relations_df["weak_candidate"] == True]
    .groupby("relation")
    .agg(
        count=("relation", "count"),
        example=("sentence", "first")
    )
    .sort_values("count", ascending=False)
)

weak_relations.head(50)

,count,example
relation,,
representar,6,"Por otro lado, siguiendo con las anécdotas que..."
tener,6,"De hecho, la ***Genealogía*** tan sólo documen..."
figurar,5,"El reparto que figura en el manuscrito, que he..."
véase,5,"De hecho, la ***Genealogía*** tan sólo documen..."
corresponder,3,"El reparto que figura en el manuscrito, que he..."
dar,2,"/ Rana.- Verá el diabro, o mala rabia / te dé,..."
hacer,2,"Así, M. L. Lobato recoge el testimonio del ent..."
indicar,2,"El reparto que figura en el manuscrito, que he..."
creer,2,Y volviéndose a mirar las ventanas donde había...
